# Data Preparation Functions

In [13]:
# importing libraries
import yfinance as yf
import pandas as pd
import requests
import time

print("Libraries imported successfully!")

Libraries imported successfully!


## Function for Actions
This function calculates the target variable - the optimal action for that day (used only for training data) by looking at how the price differs on a day 5 days before the current day. When last 5 days are reached, every day is looking at the price of the very last day, last day copies action of the previous day.

In [14]:
# function for actions (target variable)

def calculate_actions(company, simulation_year):

    # getting historical data
    historical_data = yf.download(
        company,
        start=f"{simulation_year - 2}-01-01",
        end=f"{simulation_year - 1}-12-31"
    )

    # setting uniform index
    historical_data.index = historical_data.index.tz_localize(None)

    # creating list of closing prices
    price_close = historical_data["Close"].values.flatten().tolist()

    # creating an empty list for actions
    actions = []

    # 1% percentage change threshold
    threshold = 0.01

    # looping over prices and determining action - 0 hold, 1 buy, -1 sell
    for i in range(len(price_close)):
        
        # check whether the current day can look 5 days ahead from current day and calculate percentage change
        if len(price_close) - i > 5:

            # calculating percentage change
            pct_change = ((price_close[i + 5] - price_close[i]) / price_close[i])

        # when last 5 days are reached - look at 4 days ahead, 3 days, 2 days, 1 day, and on last day do the same as day before
        elif len(price_close) - i == 5:
            
            # calculating percentage change - ((new value - old value) / old value)
            pct_change = ((price_close[i + 4] - price_close[i]) / price_close[i])
        
        # 3 days ahead
        elif len(price_close) - i == 4:
            
            # calculating percentage change 
            pct_change = ((price_close[i + 3] - price_close[i]) / price_close[i])
        
        # 2 days ahead
        elif len(price_close) - i == 3:

            # calculating percentage change 
            pct_change = ((price_close[i + 2] - price_close[i]) / price_close[i])
        
        # 1 day ahead
        elif len(price_close) - i == 2:
            
            # calculating percentage change 
            pct_change = ((price_close[i + 1] - price_close[i]) / price_close[i])
        
        # previous day's action (last day)
        else:
            
            # appending previous day's action
            actions.append(actions[-1])

            continue

        # if pct change is greater or equal than 0.01 (threshold) - buy, if less than or equal to -0.01, else hold
        if pct_change >= threshold:
            actions.append(1)
        elif pct_change <= (-threshold):
            actions.append(-1)
        else:
            actions.append(0)
        
    # returning actions as a series
    return pd.Series(actions, index=historical_data.index)


# test case
company = "AAPL"
simulation_year = 2020

# calling the function and printing the actions (list)
calculate_actions(company, simulation_year)


[*********************100%***********************]  1 of 1 completed


Date
2018-01-02    1
2018-01-03    1
2018-01-04    1
2018-01-05    1
2018-01-08    1
             ..
2019-12-23    1
2019-12-24    1
2019-12-26    0
2019-12-27    0
2019-12-30    0
Length: 502, dtype: int64

## Functions for Technical Agent

### 1. **train_data_technical_agent**
This functioon will calculate following features:
1. `Moving Averages (10-day)`: average price over 10 days
2. `Daily Price Change`: difference between stock's opening price for the day and the ending price at the end of the day
3. `Daily Price Fluctuation`: difference between the largest value of a stock for a day and the lowest value of the stock for the day 
4. `Volatility`: how greatly a stock or other asset’s prices swing around the mean price - represents "market's fear"
5. `Volatility Change`: difference between highest and lowest volatility
6. `Volume Change`: change in volume between current and previous day (number of shares traded)
7. `Action (decision)`: look 5 days ahead at the price - if the price change in 5 days from current day is >= by 1% than the current day, set curren's day to "buy", if <= current day, then sell, otherwise hold
    -  when 4 days till the end of the period, look at the price 4 days ahead, etc. (essentially look at the last day every time) and last day copies the percentage change from the day



In [15]:
# function for technical agent's training data

def train_data_technical_agent(company, simulation_year):

    # getting historical data for daily price change, volume change, and price fluctuation
    historical_data = yf.download(
        company,
        start=f"{simulation_year - 2}-01-01",
        end=f"{simulation_year - 1}-12-31",
        progress=False
    )

    # getting data for volatility
    volatility_data = yf.download(
        "^VIX",
        start=f"{simulation_year - 2}-01-01",
        end=f"{simulation_year - 1}-12-31",
        progress=False
    )
    
    # since both datframes have different index (different timezones) - setting it to the same index
    historical_data.index = historical_data.index.tz_localize(None)
    volatility_data.index = volatility_data.index.tz_localize(None)

    # creating suffixes for volatility data
    volatility_data = volatility_data.add_suffix("_vix")

    # sorting indexes for both dataframes before the merge
    historical_data = historical_data.sort_index()
    volatility_data = volatility_data.sort_index()

    # the volatility may be reported on days which stock market is not open - joining the two dataframes by index (by finding the closest day
    # to the trading day)
    historical_data = pd.merge_asof(
        historical_data,
        volatility_data,
        left_index=True,
        right_index=True,
        direction="backward"
    )

    # creating a dataframe for technical agent
    df_technical = pd.DataFrame(index=historical_data.index)


    ## moving average (10-day):

    # 10-moving average - creates nulls for days 0-9
    df_technical["moving_average_10_day"] = historical_data["Close"].rolling(window=10).mean()

    # handling nulls for moving average in days 0-9
    df_technical["moving_average_10_day"] = df_technical["moving_average_10_day"].bfill()

    
    ## price:

    # daily price change
    df_technical["daily_price_change"] = historical_data["Open"] - historical_data["Close"]

    # dialy price fluctuation
    df_technical["daily_price_fluctuation"] = historical_data["High"] - historical_data["Low"]

    
    ## volatility:
    
    # volatility
    df_technical["volatility"] = historical_data["Close_vix"]

    # volatility change
    df_technical["volatility_change"] = historical_data["High_vix"] - historical_data["Low_vix"]


    ## volume change:

    # creating a list of volume values
    volumes = historical_data["Volume"].values.flatten().tolist()
    
    # volume change
    df_technical["volume_change"] = [volumes[i] - volumes[i - 1] if i != 0 else 0 for i in range(len(volumes))]


    ## action:

    # calling the function to calculate actions and creating a column for the target variable
    df_technical["action"] = calculate_actions(company, simulation_year)

    # returning the training data dataframe
    return df_technical


# test case
company = "AAPL"
simulation_year = 2020

# calling the function and printing the test case result
x = train_data_technical_agent(company, simulation_year)
x

[*********************100%***********************]  1 of 1 completed


,moving_average_10_day,daily_price_change,daily_price_fluctuation,volatility,volatility_change,volume_change,action
Date,,,,,,,
2018-01-02,40.806045,-0.491341,0.711280,9.77,1.549999,0,1
2018-01-03,40.806045,0.070193,0.605989,9.15,0.710000,15848000,1
2018-01-04,40.806045,-0.114648,0.325222,9.22,0.390000,-28333200,1
2018-01-05,40.806045,-0.364997,0.542815,9.22,0.540000,4901600,1
2018-01-08,40.806045,0.000000,0.393076,9.52,0.570001,-12368800,1
...,...,...,...,...,...,...,...
2019-12-23,66.755762,-0.836457,0.935290,12.61,0.490000,-177406000,1
2019-12-24,67.136385,0.101246,0.474876,12.67,0.280000,-50093200,1
2019-12-26,67.597761,-1.226963,1.272764,12.65,1.030000,44642400,0


In [16]:
# distribution of actions
x.value_counts("action")

action
 1    253
-1    140
 0    109
Name: count, dtype: int64

### **2. simulation_data_technical_agent**
This function will calculate same features just like the train function, but will not include the `action` column and returns dataframe for simulation 


In [17]:
# function for technical agent's simulation data

def simulation_data_technical_agent(company, simulation_year):

    # getting historical data for daily price change, volume change, and price fluctuation
    historical_data = yf.download(
        company,
        start=f"{simulation_year}-01-01",
        end=f"{simulation_year}-12-31",
        progress=False
    )

    # getting data for volatility
    volatility_data = yf.download(
        "^VIX",
        start=f"{simulation_year}-01-01",
        end=f"{simulation_year}-12-31",
        progress=False
    )
    
    # since both datfarames have different index (different timezones) - setting it to the same index and resetting the index
    historical_data.index = historical_data.index.tz_localize(None)
    volatility_data.index = volatility_data.index.tz_localize(None)

    # creating suffixes for volatility data
    volatility_data = volatility_data.add_suffix("_vix")

    # sorting indexes for both dataframes before the merge
    historical_data = historical_data.sort_index()
    volatility_data = volatility_data.sort_index()

    # the volatility may be reported on days which stock market is not open - joining the two dataframes by index (by finding the closest day
    # to the trading day)
    historical_data = pd.merge_asof(
        historical_data,
        volatility_data,
        left_index=True,
        right_index=True,
        direction="backward"
    )

    # creating a dataframe for technical agent
    df_technical = pd.DataFrame(index=historical_data.index)

    ## moving average (10-day):

    # 10-moving average - creates nulls for days 0-9
    df_technical["moving_average_10_day"] = historical_data["Close"].rolling(window=10).mean()

    # handling nulls for moving average in days 0-9
    df_technical["moving_average_10_day"] = df_technical["moving_average_10_day"].bfill()

    
    ## price:

    # daily price change
    df_technical["daily_price_change"] = historical_data["Open"] - historical_data["Close"]

    # dialy price fluctuation
    df_technical["daily_price_fluctuation"] = historical_data["High"] - historical_data["Low"]

    
    ## volatility:
    
    # volatility
    df_technical["volatility"] = historical_data["Close_vix"]

    # volatility change
    df_technical["volatility_change"] = historical_data["High_vix"] - historical_data["Low_vix"]


    ## volume change:

    # creating a list of volume values
    volumes = historical_data["Volume"].values.flatten().tolist()
    
    # volume change
    df_technical["volume_change"] = [volumes[i] - volumes[i - 1] if i != 0 else 0 for i in range(len(volumes))]

    # return simulation dataframe
    return df_technical


# test case
company = "AAPL"
simulation_year = 2020

# calling the function and printing the test case result
x = simulation_data_technical_agent(company, simulation_year)
x

,moving_average_10_day,daily_price_change,daily_price_fluctuation,volatility,volatility_change,volume_change
Date,,,,,,
2020-01-02,73.764877,-0.990735,1.304102,12.470000,1.300000,0
2020-01-03,73.764877,-0.067495,0.983496,14.020000,3.070001,10842400
2020-01-06,73.764877,-1.448728,1.737996,13.850000,2.849999,-27935600
2020-01-07,73.764877,0.349524,0.824400,13.790000,1.070000,-9515200
2020-01-08,73.764877,-1.453554,1.754872,13.450000,2.410000,23207200
...,...,...,...,...,...,...
2020-12-23,123.469105,1.167048,1.604689,23.309999,1.550001,-80681100
2020-12-24,124.318135,-0.632147,2.295200,21.530001,1.440001,-33293600
2020-12-28,125.706925,-2.625861,3.724839,21.700001,0.970001,69556100


## Functions for Health Agent

### 1. Function for Retreiving the Data and Putting to the Right Format
Here, I am using `Alpha Vantage` API for retrieving neccessary data for health agent. The API is free, but has some limits on the number of times you can call it using the **FREE** version. The API data comes in as a `string` for all features. This function will create one dataframe with all neccessary features for the `train_data_health_agent` function.

In [18]:
# function for retreiving API data and putting it to the right format

def get_health_agent_data(company, simulation_year):

    # setting the API KEY
    API = "G6OTTEU16IXL124Y" # or G6OTTEU16IXL124Y K34H6QMPMV2QNSC0

    # setting the company
    COMPANY = company

    # 1. retrieving income statements
    url_income_statements= f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={COMPANY}&apikey={API}"
    r_income_statements = requests.get(url_income_statements)
    income_statements = r_income_statements.json()
    time.sleep(20) # to avoid crash because of the API call limit per minute

    # 2. retrieving balance sheets
    url_balance_sheets = f"https://www.alphavantage.co/query?function=BALANCE_SHEET&symbol={COMPANY}&apikey={API}"
    r_balance_sheets = requests.get(url_balance_sheets)
    balance_sheets = r_balance_sheets.json()
    time.sleep(20) # to avoid crash because of the API call limit per minute
    
    # 3. retreiving EPS (earnings-per-share)
    url_eps = f"https://www.alphavantage.co/query?function=EARNINGS&symbol={COMPANY}&apikey={API}"
    r_eps = requests.get(url_eps)
    eps = r_eps.json()
    time.sleep(20) # to avoid crash because of the API call limit per minute

    # 4. retreiving stock's price
    stock_price = yf.download(
        COMPANY,
        start=f"{simulation_year - 3}-01-01",
        end=f"{simulation_year}-12-31",
        progress=False
    )   

    # extracting the quaterly reports from income statements, balance sheets, EPS, and converting it to invidual dataframes
    df_income_statements = pd.DataFrame(income_statements["quarterlyReports"])
    df_balance_sheets = pd.DataFrame(balance_sheets["quarterlyReports"])
    df_eps = pd.DataFrame(eps["quarterlyEarnings"])


    # extracting stock price and converting to a dataframe stock price, 
    df_price = pd.DataFrame(stock_price["Close"])

    ## setting indexes of all dataframes to datetime object and universal time:

    # income statements
    df_income_statements["fiscalDateEnding"] = pd.to_datetime(df_income_statements["fiscalDateEnding"])
    df_income_statements.set_index("fiscalDateEnding", inplace=True)
    df_income_statements.index = df_income_statements.index.tz_localize(None)

    # balance sheets
    df_balance_sheets["fiscalDateEnding"] = pd.to_datetime(df_balance_sheets["fiscalDateEnding"])
    df_balance_sheets.set_index("fiscalDateEnding", inplace=True)
    df_balance_sheets.index = df_balance_sheets.index.tz_localize(None)

    # EPS
    df_eps["fiscalDateEnding"] = pd.to_datetime(df_eps["fiscalDateEnding"])
    df_eps.set_index("fiscalDateEnding", inplace=True)
    df_eps.index = df_eps.index.tz_localize(None)

    # stock price
    df_price.index = df_price.index.tz_localize(None)

    
    ## selecting columns which to include in each dataframe
    df_income_statements = df_income_statements[["totalRevenue", "netIncome"]]
    df_balance_sheets = df_balance_sheets[["totalLiabilities", "totalShareholderEquity", "totalCurrentAssets", "totalCurrentLiabilities"]]
    df_eps = df_eps[["reportedEPS", "estimatedEPS"]]

    # sorting indexes of all dataframes for the merge
    df_price = df_price.sort_index()
    df_income_statements = df_income_statements.sort_index()
    df_balance_sheets = df_balance_sheets.sort_index()
    df_eps = df_eps.sort_index()

    # joining quaterly reports to one dataframe - if any dates differ, outer join ensures nothing is lost 
    df_quaterly_reports = df_income_statements.join([df_balance_sheets, df_eps], how="outer")

    # merging the stock price data and the quaterly reports data
    health_data = pd.merge_asof(
        df_price,
        df_quaterly_reports,
        left_index=True,
        right_index=True,
        direction="backward"
    )

    # renaming the column for stock price (currently is named afer the company's ticker)
    health_data.rename(columns={COMPANY: "Close"}, inplace=True)

    # the dataframe has 4 years of data now -> the quaterly reports data is joined to stock's price index (trading days) based on the quaterly
    # reports date, but these often are not the same (when the quaterly reports are filed, the stock market is often not open) -> that is why
    # I used the ".merge_asof()" method to join the quaterly reports to the closest day before the quaterly reports were filed and the stock market
    # was still open
    health_data = health_data.ffill()

    # converting all the columns to numbers -> the API returns values as strings(objects)
    health_data = health_data.apply(pd.to_numeric, errors="coerce")

    return health_data

# test case
company = "AAPL"
simulation_year = 2020

# calling the function and printing the test dataframe
x = get_health_agent_data(company, simulation_year)
x

,Close,totalRevenue,netIncome,totalLiabilities,totalShareholderEquity,totalCurrentAssets,totalCurrentLiabilities,reportedEPS,estimatedEPS
Date,,,,,,,,,
2017-01-03,26.745853,78351000000,17891000000,198751000000,132390000000,103332000000,84130000000,0.84,0.8025
2017-01-04,26.715921,78351000000,17891000000,198751000000,132390000000,103332000000,84130000000,0.84,0.8025
2017-01-05,26.851784,78351000000,17891000000,198751000000,132390000000,103332000000,84130000000,0.84,0.8025
2017-01-06,27.151129,78351000000,17891000000,198751000000,132390000000,103332000000,84130000000,0.84,0.8025
2017-01-09,27.399820,78351000000,17891000000,198751000000,132390000000,103332000000,84130000000,0.84,0.8025
...,...,...,...,...,...,...,...,...,...
2020-12-23,127.364143,64698000000,12673000000,258549000000,65339000000,143713000000,105392000000,0.73,0.7000
2020-12-24,128.346405,64698000000,12673000000,258549000000,65339000000,143713000000,105392000000,0.73,0.7000
2020-12-28,132.936783,64698000000,12673000000,258549000000,65339000000,143713000000,105392000000,0.73,0.7000


### 2. **train_data_health_agent**
This function creates following features:
1. `Revenue Growth`: an increase/decrease in revenue over time
2. `Net Income`: profit a company retains after all expenses, taxes, and deductions
3. `Profit Margin`: a percentage of profit earned by a company in relation to its revenue
4. `Debt-to-equity Ratio`: a ratio comparing company's liabilities and its shareholder equity
5. `Current Ratio`: a liqudity ratio measuring whether the company has enough resources/can meet its short-term obligations
6.  `Earnings per Share`: indication of company's curren financial strength
7. `Price-to-earnings Ratio`: a ratio telling how much investors are paying for a dollar of a company's earnings
8. `Action`: an optimal action determined by the function for actions

In [19]:
# function for training data for health_agent

def train_data_health_agent(company, simulation_year, **kwargs):

    # "kwargs" ensure there is no need to define training years when calling the function
    if kwargs:
        training_year_1 = simulation_year
        training_year_2 = simulation_year
    else:
        training_year_1 = simulation_year - 2
        training_year_2 = simulation_year - 1

    # setting the company and simulation year constants
    COMPANY = company
    SIMULATION_YEAR = simulation_year

    # calling the function to retrieve data for the health agent and storing the dataframe
    health_data = get_health_agent_data(COMPANY, SIMULATION_YEAR)

    # downloading same data technical agent uses - for indexing purposes
    technical_agent_data = yf.download(
        company,
        start=f"{training_year_1}-01-01",
        end=f"{training_year_2}-12-31",
        progress=False
    )

    # setting uniform index
    technical_agent_data.index = technical_agent_data.index.tz_localize(None)

    # creating empty dataframe for the final training dataframe and using technical agent's data for indexing to ensure same length
    df_health_agent = pd.DataFrame(index=technical_agent_data.index)

    ## creating features:

    # 1. revenue growth
    
    # extracting quarter revenues
    quarter_revenues = health_data["totalRevenue"].drop_duplicates()

    # calculating the revenue growth
    revenue_growth = quarter_revenues.pct_change()

    # creating a column for revenue growth and using the "revenue_growth" values
    df_health_agent["revenue_growth"] = revenue_growth

    # the column has revenue_growth for all quarters except the very first quarter of the dataset(the year prior to training years) - fill with 0, else use forward fill
    df_health_agent["revenue_growth"] = df_health_agent["revenue_growth"].ffill().fillna(0)

    # 2. net income
    df_health_agent["net_income"] = health_data["netIncome"]

    # 3. profit margin
    df_health_agent["profit_margin"] = health_data["netIncome"] / health_data["totalRevenue"]

    # 4. debt-to-equity ration
    df_health_agent["debt_to_equity_ratio"] = health_data["totalLiabilities"] / health_data["totalShareholderEquity"]

    # 5. current ration
    df_health_agent["current_ratio"] = health_data["totalCurrentAssets"] / health_data["totalCurrentLiabilities"]

    # 6. earnings per share (EPS)
    df_health_agent["EPS"] = health_data["reportedEPS"]

    # 7. price-to-earnings ratio
    df_health_agent["price_to_earnings_ratio"] = health_data["Close"] / (health_data["reportedEPS"] * 4)

    # 8. action - if kwargs are provided, the function creates simulation dataset and actions are not needed
    if kwargs:
        
        # returning the dataframe without actions - simulation
        return df_health_agent.loc[str(training_year_1):str(training_year_2)]
    
    # if kwargs are not provided, the function creates training dataset and actions are provided
    else:
        
        # adding an action column
        df_health_agent["action"] = calculate_actions(COMPANY, SIMULATION_YEAR)

        # returning final dataframe with years for training
        return df_health_agent.loc[str(training_year_1):str(training_year_2)]

# test case
company = "AAPL"
simulation_year = 2020

# calling the function and printing the test dataframe
x = train_data_health_agent("AAPL", 2020)
x

[*********************100%***********************]  1 of 1 completed


,revenue_growth,net_income,profit_margin,debt_to_equity_ratio,current_ratio,EPS,price_to_earnings_ratio,action
Date,,,,,,,,
2018-01-02,0.679245,20065000000,0.227255,1.901547,1.242011,0.9725,10.360972,1
2018-01-03,0.679245,20065000000,0.227255,1.901547,1.242011,0.9725,10.359167,1
2018-01-04,0.679245,20065000000,0.227255,1.901547,1.242011,0.9725,10.407285,1
2018-01-05,0.679245,20065000000,0.227255,1.901547,1.242011,0.9725,10.525776,1
2018-01-08,0.679245,20065000000,0.227255,1.901547,1.242011,0.9725,10.486678,1
...,...,...,...,...,...,...,...,...
2019-12-23,0.190135,13686000000,0.213710,2.741004,1.540126,0.7600,22.519503,1
2019-12-24,0.190135,13686000000,0.213710,2.741004,1.540126,0.7600,22.540911,1
2019-12-26,0.190135,13686000000,0.213710,2.741004,1.540126,0.7600,22.988134,0


In [20]:
# distribution of the action variable
x.value_counts("action")

action
 1    253
-1    140
 0    109
Name: count, dtype: int64

### 3. **simulation_data_health_agent**
This function does the same thing, just like the `train_data_health_agent` function, but returns a dataframe for the simulation year without the `action` column.

In [21]:
# function for simulation data for health_agent

def simulation_data_health_agent(company, simulation_year):

    # setting the company and simulation year constants
    COMPANY = company
    SIMULATION_YEAR = simulation_year

    # calling the train_data_health_agent function
    df_health_agent = train_data_health_agent(COMPANY, SIMULATION_YEAR, simulation=True)

    # returning the final dataframe for simulation
    return df_health_agent

# test case
company = "AAPL"
simulation_year = 2020

# calling the function and printing the test dataframe
x = simulation_data_technical_agent("AAPL", 2020)
x

,moving_average_10_day,daily_price_change,daily_price_fluctuation,volatility,volatility_change,volume_change
Date,,,,,,
2020-01-02,73.764877,-0.990735,1.304102,12.470000,1.300000,0
2020-01-03,73.764877,-0.067495,0.983496,14.020000,3.070001,10842400
2020-01-06,73.764877,-1.448728,1.737996,13.850000,2.849999,-27935600
2020-01-07,73.764877,0.349524,0.824400,13.790000,1.070000,-9515200
2020-01-08,73.764877,-1.453554,1.754872,13.450000,2.410000,23207200
...,...,...,...,...,...,...
2020-12-23,123.469105,1.167048,1.604689,23.309999,1.550001,-80681100
2020-12-24,124.318135,-0.632147,2.295200,21.530001,1.440001,-33293600
2020-12-28,125.706925,-2.625861,3.724839,21.700001,0.970001,69556100
